In [6]:
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/Just_Advisor_Ai"
DATA_PATH = f"{BASE}/data/argument_dataset_indian_law.jsonl"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
BASE = "/content/drive/MyDrive/Just_Advisor_Ai"
DATA_PATH = f"{BASE}/data/argument_dataset_indian_law.jsonl"


In [8]:
!pip install transformers datasets accelerate torch


In [9]:
import re
import json

bad_path = "/content/drive/MyDrive/Just_Advisor_Ai/data/argument_dataset_indian_law.jsonl"
fixed_path = "/content/drive/MyDrive/Just_Advisor_Ai/data/argument_dataset_fixed.jsonl"

fixed = []
bad_count = 0

with open(bad_path, "r", encoding="utf-8", errors="ignore") as f:
    raw = f.read()

# Split objects using pattern
objects = re.findall(r'\{.*?\}', raw, re.DOTALL)

for obj_str in objects:
    try:
        # Fix common broken quotes like accused"s → accused's
        obj_str = re.sub(r'(\w)"s', r"\1's", obj_str)

        # Ensure valid JSON quotes
        obj = json.loads(obj_str)
        fixed.append(obj)

    except Exception as e:
        bad_count += 1

with open(fixed_path, "w", encoding="utf-8") as f:
    for obj in fixed:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Fixed dataset saved to:", fixed_path)
print("✅ Valid samples:", len(fixed))
print("❌ Skipped corrupted lines:", bad_count)


✅ Fixed dataset saved to: /content/drive/MyDrive/Just_Advisor_Ai/data/argument_dataset_fixed.jsonl
✅ Valid samples: 900
❌ Skipped corrupted lines: 0


In [10]:
DATA_PATH = "/content/drive/MyDrive/Just_Advisor_Ai/data/argument_dataset_fixed.jsonl"


In [11]:
from datasets import load_dataset

dataset = load_dataset("json", data_files=DATA_PATH, split="train")
dataset


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['text', 'label'],
    num_rows: 900
})

In [12]:
labels = ["Claim", "Evidence", "LegalRule", "Rebuttal", "Conclusion"]
label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}

def encode_labels(x):
    x["label"] = label2id[x["label"]]
    return x

dataset = dataset.map(encode_labels)


Map:   0%|          | 0/900 [00:00<?, ? examples/s]

In [13]:
dataset = dataset.train_test_split(test_size=0.1)
train_ds = dataset["train"]
val_ds = dataset["test"]


In [14]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

train_ds.set_format("torch", columns=["input_ids","attention_mask","label"])
val_ds.set_format("torch", columns=["input_ids","attention_mask","label"])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/810 [00:00<?, ? examples/s]

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

In [15]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "nlpaueb/legal-bert-base-uncased",
    num_labels=5,
    id2label=id2label,
    label2id=label2id
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [16]:
!pip install -U transformers accelerate datasets


In [17]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Just_Advisor_Ai/models/legalbert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    fp16=True,
)


In [18]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer
)

trainer.train()


/tmp/ipython-input-1713228816.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,1.335200,0.724415
2,0.489500,0.137375
3,0.104300,0.055035


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=153, training_loss=0.6317008087058472, metrics={'train_runtime': 3323.959, 'train_samples_per_second': 0.731, 'train_steps_per_second': 0.046, 'total_flos': 159844271546880.0, 'train_loss': 0.6317008087058472, 'epoch': 3.0})

In [19]:
SAVE_PATH = "/content/drive/MyDrive/Just_Advisor_Ai/models/legalbert/final"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model saved to:", SAVE_PATH)


Model saved to: /content/drive/MyDrive/Just_Advisor_Ai/models/legalbert/final


In [20]:
SAVE_PATH = "/content/drive/MyDrive/Just_Advisor_Ai/models/legalbert/final"

import os
os.makedirs(SAVE_PATH, exist_ok=True)

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Saved to:", SAVE_PATH)
print("Files:", os.listdir(SAVE_PATH))


Saved to: /content/drive/MyDrive/Just_Advisor_Ai/models/legalbert/final
Files: ['config.json', 'model.safetensors', 'tokenizer_config.json', 'special_tokens_map.json', 'vocab.txt', 'tokenizer.json']


In [22]:
import torch

def predict(sentence):
    inputs = tokenizer(sentence, return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
    return id2label[logits.argmax().item()]

predict("The accused dishonestly induced the complainant to deliver property.")


'Claim'